# 3D Floor Plan Visualizer
Interactive 3D viewer for 2BHK and 3BHK layouts.
- **Rooms** : coloured flat floor polygons with labels
- **Columns** : vertical lines (red = perimeter, blue = internal)
- **Beams** : horizontal lines (orange = perimeter, cyan = internal)
- Each level stacks at z = 0, 3, 6, 9 m


## Setup
Install dependencies before running:
```bash
pip install plotly
``` 

### Import  

In [ ]:
import json, os
import plotly.graph_objects as go
import numpy as np
from IPython.display import display
                

In [12]:
# ── File paths: update these to point at your multilevel JSON files ──
FILES = {    
    "2BHK Op3": "../data/layout_2bhk_op3_multilevel.json",
    "3BHK Op3": "../data/layout_3bhk_op3_multilevel.json",
}

LEVEL_HEIGHT = 3.0   # metres between floors

# Room colour palette (cycles if more rooms than colours)
ROOM_COLOURS = [
    "rgba(173,216,230,0.45)",  # light blue
    "rgba(144,238,144,0.45)",  # light green
    "rgba(255,228,181,0.45)",  # moccasin
    "rgba(216,191,216,0.45)",  # thistle
    "rgba(255,182,193,0.45)",  # light pink
    "rgba(176,224,230,0.45)",  # powder blue
    "rgba(255,239,186,0.45)",  # light yellow
    "rgba(204,255,204,0.45)",  # mint
    "rgba(255,204,229,0.45)",  # blush
    "rgba(204,229,255,0.45)",  # sky
    "rgba(229,204,255,0.45)",  # lavender
    "rgba(255,218,185,0.45)",  # peach
    "rgba(200,230,200,0.45)",  # sage
]

def load_layout(path):
    with open(path) as f:
        return json.load(f)

def polygon_centroid(pts):
    xs = [p[0] for p in pts[:-1]]
    ys = [p[1] for p in pts[:-1]]
    return sum(xs)/len(xs), sum(ys)/len(ys)

print("Helpers loaded.")


Helpers loaded.


In [49]:
def build_traces(layout_data, layout_name):
    traces = []
    levels = layout_data.get("levels", {})
    level_keys = sorted(levels.keys())

    for li, lk in enumerate(level_keys):
        z_base = li * LEVEL_HEIGHT
        level = levels[lk]

        # ROOMS (flat polygons at z_base)
        rooms = level.get("rooms", [])
        for ri, room in enumerate(rooms):
            geom = room["geometry"]
            xs = [p[0] for p in geom]
            ys = [p[1] for p in geom]
            colour = ROOM_COLOURS[ri % len(ROOM_COLOURS)]
            cx, cy = polygon_centroid(geom)

            pts = list(zip(xs[:-1], ys[:-1]))
            n = len(pts)
            vx = [p[0] for p in pts] + [cx]
            vy = [p[1] for p in pts] + [cy]
            vz = [z_base] * (n + 1)
            ci_idx = n
            i_tri, j_tri, k_tri = [], [], []
            for t in range(n):
                i_tri.append(ci_idx)
                j_tri.append(t)
                k_tri.append((t + 1) % n)

            traces.append(go.Mesh3d(
                x=vx, y=vy, z=vz,
                i=i_tri, j=j_tri, k=k_tri,
                color=colour,
                opacity=0.5,
                flatshading=True,
                lighting=dict(ambient=1.0),
                contour=go.mesh3d.Contour(show=False),
                name=f"{room['name']} ({lk})",
                showlegend=(li == 0),
                hovertemplate=(
                    f"<b>{room['name']}</b><br>"
                    f"Level: {lk}<br>"
                    f"Area: {room['attributes'].get('area', 'N/A')} m2<extra></extra>"
                ),
            ))

            # room label
            traces.append(go.Scatter3d(
                x=[cx], y=[cy], z=[z_base + 0.05],
                mode="text",
                text=[room["name"]],
                textfont=dict(size=9, color="white"),
                showlegend=False,
                hoverinfo="skip",
            ))

        # STRUCTURE: columns and beams
        structure = level.get("structure", [])
        for elem in structure:
            geom = elem["geometry"]
            etype = elem["attributes"].get("type", "internal")
            is_col = elem["name"].startswith("Col_")
            is_beam = elem["name"].startswith("Beam_")

            if is_col:
                x0, y0 = geom[0]
                col_colour = "#E63946" if etype == "perimeter" else "#457B9D"

                # column vertical line
                traces.append(go.Scatter3d(
                    x=[x0, x0], y=[y0, y0], z=[z_base, z_base + LEVEL_HEIGHT],
                    mode="lines",
                    line=dict(color=col_colour, width=6),
                    name=f"{'Perimeter' if etype=='perimeter' else 'Internal'} Column",
                    legendgroup=f"col_{etype}",
                    showlegend=False,
                    hovertemplate=f"<b>{elem['name']}</b><br>Type: {etype}<br>Level: {lk}<extra></extra>",
                ))

                # base box only for level 1 (li == 0), 0.5m tall going downward
                if li == 0:
                    s = 0.5
                    h = 0.5
                    bx = [x0-s, x0+s, x0+s, x0-s, x0-s]
                    by = [y0-s, y0-s, y0+s, y0+s, y0-s]

                    # top ring at z_base
                    traces.append(go.Scatter3d(
                        x=bx, y=by, z=[z_base] * 5,
                        mode="lines",
                        line=dict(color=col_colour, width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    ))
                    # bottom ring at z_base - h
                    traces.append(go.Scatter3d(
                        x=bx, y=by, z=[z_base - h] * 5,
                        mode="lines",
                        line=dict(color=col_colour, width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    ))
                    # 4 vertical edges going down
                    for ex, ey in [(x0-s, y0-s), (x0+s, y0-s), (x0+s, y0+s), (x0-s, y0+s)]:
                        traces.append(go.Scatter3d(
                            x=[ex, ex], y=[ey, ey], z=[z_base, z_base - h],
                            mode="lines",
                            line=dict(color=col_colour, width=2),
                            showlegend=False,
                            hoverinfo="skip",
                        ))

                # column label
                traces.append(go.Scatter3d(
                    x=[x0], y=[y0],
                    z=[z_base + LEVEL_HEIGHT / 2],
                    mode="text",
                    text=[f"Lv{li+1}_{elem['id']}"],
                    textfont=dict(size=7, color="#E63946"),
                    showlegend=False,
                    hoverinfo="skip",
                ))

            elif is_beam:
                if len(geom) >= 2:
                    x0, y0 = geom[0]
                    x1, y1 = geom[1]
                    beam_colour = "#F4A261" if etype == "perimeter" else "#2EC4B6"

                    # beam horizontal line
                    traces.append(go.Scatter3d(
                        x=[x0, x1], y=[y0, y1],
                        z=[z_base + LEVEL_HEIGHT, z_base + LEVEL_HEIGHT],
                        mode="lines",
                        line=dict(color=beam_colour, width=4),
                        name=f"{'Perimeter' if etype=='perimeter' else 'Internal'} Beam",
                        legendgroup=f"beam_{etype}",
                        showlegend=False,
                        hovertemplate=f"<b>{elem['name']}</b><br>Type: {etype}<br>Length: {elem['attributes'].get('length','N/A')} m<br>Level: {lk}<extra></extra>",
                    ))
                    # beam label
                    traces.append(go.Scatter3d(
                        x=[(x0 + x1) / 2], y=[(y0 + y1) / 2],
                        z=[z_base + LEVEL_HEIGHT + 0.05],
                        mode="text",
                        text=[f"Lv{li+1}_{elem['id']}"],
                        textfont=dict(size=7, color="#F4A261"),
                        showlegend=False,
                        hoverinfo="skip",
                    ))

    # Legend dummy traces
    traces += [
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
            line=dict(color="#E63946", width=6), name="Perimeter Column", legendgroup="col_perimeter"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
            line=dict(color="#457B9D", width=6), name="Internal Column", legendgroup="col_internal"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
            line=dict(color="#F4A261", width=4), name="Perimeter Beam", legendgroup="beam_perimeter"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
            line=dict(color="#2EC4B6", width=4), name="Internal Beam", legendgroup="beam_internal"),
    ]

    return traces

print("Trace builder ready.")

Trace builder ready.


In [46]:
def show_layout(label, filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return

    data = load_layout(filepath)
    traces = build_traces(data, label)

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=f"3D Floor Plan Visualizer  |  {label}", font=dict(size=16)),
       scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(eye=dict(x=1.5, y=-2.0, z=1.2)),
        bgcolor="#1a1a2e",
        ),
        legend=dict(
            title="Legend",
            bgcolor="rgba(0,0,0,0.5)",
            bordercolor="rgba(255,255,255,0.2)",
            borderwidth=1,
            font=dict(color="#ffffff"),
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=700,
        paper_bgcolor="#1a1a2e",
        font=dict(color="#eee"),
    )
    fig.show()

print("Plot function ready.")


Plot function ready.


### Check files 

In [47]:
for label, path in FILES.items():
    exists = os.path.exists(path)
    status = "FOUND" if exists else "NOT FOUND"
    print(f"{status} | {label} -> {path}")

FOUND | 2BHK Op3 -> ../data/layout_2bhk_op3_multilevel.json
FOUND | 3BHK Op3 -> ../data/layout_3bhk_op3_multilevel.json


## View Layouts
Run each cell below to open the interactive 3D viewer for each file.

In [50]:
show_layout("2BHK Op3", FILES["2BHK Op3"])

In [51]:
show_layout("3BHK Op3", FILES["3BHK Op3"])